In [ ]:
import polars as pl
from pathlib import Path
from openhexa.sdk import workspace
from openhexa.toolbox.dhis2 import DHIS2
from openhexa.toolbox.dhis2.dataframe import get_organisation_units
from sqlalchemy import create_engine
import gc

from pnlp_utils import log_msg, push_dataframe_to_db, check_partition_exist, upsert_partition, create_partition_indexes

In [ ]:
# Parameters 

YEAR = 2026

### Columns and mappings

In [ ]:
# Expected columns
columns_expected = [
    "aggregation_temporelle",
    "period", 
    "annee",
    "id_name",    
    "province", 
    "zone_de_sante", 
    "aire_de_sante",
    "fosa",
    "fosa_dhis2_id",
    "avb_least_type",
    "avb_absolute",
    "total_fosa"
]

# id_name filter list
id_names_to_keep = [
    "Cas Confirme Traité",
    "Cas Graves Confirmés",
    "Cas Graves Traités",
    "Cas Présumés",
    "Cas Présumés Traités",
    "Cas Simples Confirmés",
    "Cas Simples Traité Pn",
    "Cas Suspects Teste",
    "Fe Enceinte Mild",
    "Incidence Annuelle",
    "Letalite Hospi",
    "Mild Cps",
    "Mortalite Prop",
    "Moustiquaire Imprégné D'Insecticide (Mild)",
    "Positivite Tdr",
    "Positivite Tdr Ge",
    "Proportion Sp2",
    "Proportion Sp3",
    "Taux Mortalite",
    "Tdrs Réalisés",
    "Total Cas Palu",
]

### Helper functions

In [ ]:
# Aggregations logic for indicators_nd table 
def run_availability_aggregations_fosa(df_year: pl.DataFrame) -> pl.DataFrame:

    # Spatial & temporal hierarchies    
    spatial_name =  "Fosa"
    spatial_cols = ["LEVEL_1_NAME", "LEVEL_2_NAME", "LEVEL_3_NAME", "LEVEL_4_NAME", "LEVEL_5_NAME", "LEVEL_5_ID"]

    temporal_levels = {
        "Année": "Year_period",
        "Semestre": "Semester_period",
        "Trimestre": "Quarter_period",
        "Mois": "Month_period",
    }
    
    standard_cols = [
        "LEVEL_1_NAME", "LEVEL_2_NAME", "LEVEL_3_NAME", "LEVEL_4_NAME", "LEVEL_5_NAME", "LEVEL_5_ID",
        "Aggregation_temporelle", "Period", "Annee",
        "ID_NAME", "Province", "Zone_de_sante", "Aire_de_Sante", "fosa",
        "AVB_LEAST_TYPE", "AVB_ABSOLUTE", "TOTAL_FOSA"        
    ]
    
    # Add year and period columns    
    print("Creating period columns..")
    df_pl = df_year.with_columns(
        pl.col("PERIOD").cast(pl.Utf8).str.strptime(pl.Date, "%Y%m").alias("Agg_date"),
        ).with_columns(
                pl.col("Agg_date").dt.year().cast(pl.Utf8).alias("Annee"),
            pl.col("Agg_date").dt.year().cast(pl.Utf8).alias("Year_period"),
            (pl.col("Agg_date").dt.year().cast(pl.Utf8) + " S" + (((pl.col("Agg_date").dt.month() - 1) // 6 + 1).cast(pl.Utf8))).alias("Semester_period"),
            (pl.col("Agg_date").dt.year().cast(pl.Utf8) + " T" + (((pl.col("Agg_date").dt.month() - 1) // 3 + 1).cast(pl.Utf8))).alias("Quarter_period"),
            pl.col("Agg_date").dt.strftime("%Y-%m").alias("Month_period"),
        )
    
    agg_list_pl = []    
    print(f"Processing → Spatial: {spatial_name}")
    for temporal_name, temporal_col in temporal_levels.items():
        print(f"Processing → Temporal: {temporal_name}")
        
        # Polars groupby + aggregation 
        group_cols = spatial_cols + ["ID_NAME", temporal_col, "Annee"]    
        agg_df = (
            df_pl.group_by(group_cols, maintain_order=True)
                 .agg([
                     pl.sum("AVB_LEAST_TYPE").alias("AVB_LEAST_TYPE"),
                     pl.sum("AVB_ABSOLUTE").alias("AVB_ABSOLUTE"),
                     pl.sum("TOTAL FOSA").alias("TOTAL_FOSA")
                 ])
        )
             
        # Ensure all spatial columns exist with correct type (Utf8)
        for col in ["LEVEL_1_NAME", "LEVEL_2_NAME", "LEVEL_3_NAME", "LEVEL_4_NAME", "LEVEL_5_NAME", "LEVEL_5_ID"]:
            if col not in agg_df.columns:
                agg_df = agg_df.with_columns(pl.lit(None, dtype=pl.Utf8).alias(col))
    
        # Metadata 
        agg_df = agg_df.with_columns([
            # pl.lit(spatial_name).alias("Aggregation_spatiale"),
            pl.lit(temporal_name).alias("Aggregation_temporelle"), 
            pl.col(temporal_col).alias("Period"),
            pl.col("ID_NAME")
              .str.replace_all("_", " ")   # replace all underscores
              .str.to_lowercase()  
              .str.to_titlecase()                 
              .alias("ID_NAME")
        ])
              
        # Reorder agg_df and append
        agg_df = agg_df.select([col for col in standard_cols if col in agg_df.columns])
        agg_list_pl.append(agg_df)
            
    # Concatenate all results 
    df_year_agg_pl = pl.concat(agg_list_pl)  
    
    df_year_agg_pl = df_year_agg_pl.rename({
        "LEVEL_2_NAME": "Province",
        "LEVEL_3_NAME": "Zone_de_sante",
        "LEVEL_4_NAME": "Aire_de_sante",
        "LEVEL_5_NAME": "Fosa",
        "LEVEL_5_ID":   "Fosa_dhis2_id"
    })
    df_year_agg_pl = df_year_agg_pl.drop("LEVEL_1_NAME")  # drop
    df_year_agg_pl = df_year_agg_pl.rename({c: c.lower() for c in df_year_agg_pl.columns})  # cols to lower    
    return df_year_agg_pl


### Load SNIS pyramid metadata level 5  


In [ ]:
# Load pyramid metadata
dhis2_client = DHIS2(workspace.get_connection("drc-snis"))
pyramid_metadata = get_organisation_units(dhis2_client)
pyramid_metadata_selection = pyramid_metadata.filter(pl.col("level") == 5).drop(["geometry", "name", "level", "opening_date", "closed_date"])
print(pyramid_metadata_selection.shape)
pyramid_metadata_selection.head(2)

## Start processing

In [ ]:
# target table name
target_table = "availability_agg_fosa"

# Availability file path
availability_data_path = Path(workspace.files_path) / "pnlp-tdb-pipeline" / "data" / "interim" / "availability"

# Database Connection 
engine = create_engine(workspace.database_url)
print(f"Connected to: {engine.url.host}")

# We can run for a selection of years if needed..
target_years = [YEAR] 
# target_years = [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
log_msg(f"Running for year(s): {target_years}")

### Load data to DB  

- Loop over the target years
- Load from /pnlp-tdb-pipeline/data/interim/availability/*  
- Enrich metadata
- Push to db


In [ ]:
for target_year in target_years: 
    log_msg(f"Processing '{target_table}' for year: {target_year}")
    
    availability_fname = availability_data_path / f"indicators_availability_level_5_month_newformat_{target_year}.feather"
    if not availability_fname.exists():
        log_msg(f"Availability file year {target_year} not fund, process skipped!", "warning")
        continue
        
    # Load file
    availability_level_5 = pl.read_ipc(availability_data_path / f"indicators_availability_level_5_month_newformat_{target_year}.feather", memory_map=False)
    
    # Enrich with metadata
    availability_level_5_merged = availability_level_5.select([
            "ID_NAME", "PERIOD", "OU_UID", "AVB_LEAST_TYPE", "AVB_ABSOLUTE", "TOTAL FOSA"
        ]).join(
            pyramid_metadata_selection,
            how="inner",
            left_on="OU_UID",
            right_on="id"
        ).drop("OU_UID")    
    availability_level_5_merged = availability_level_5_merged.rename({col: col.upper() for col in availability_level_5_merged.columns})  # to upper
    
    # Data
    df_year = availability_level_5_merged.clone()
    
    # create temporal aggregations
    df_year_agg = run_availability_aggregations_fosa(df_year) 
    assert df_year_agg.height == df_year_agg.unique().height, "Duplicated values found!" # quick check!
    
    print("Check expected columns..", end=" ")
    df_year_agg = df_year_agg.select(columns_expected)
    print("OK!")
    
    print("Applying id_name filter by name..", end=" ")
    df_year_agg = df_year_agg.filter(pl.col("id_name").is_in(id_names_to_keep))
    df_year_agg = df_year_agg.with_columns(pl.col("annee").cast(pl.Int32))
    print("OK!")
    
    log_msg("Pushing aggregated data to db..")
    tmp_table = f"{target_table}_tmp"
    push_dataframe_to_db(
        df=df_year_agg, 
        table_name=tmp_table, 
        engine=engine, 
        push_behaviour="replace",
        chunksize=500000
    )   
    
    # update partition
    upsert_partition(engine, target_table, tmp_table, target_year)
    
    log_msg("Creating indexes..")
    index_columns = ["aggregation_temporelle", "province", "zone_de_sante", "aire_de_sante", "fosa", "period"]
    create_partition_indexes(engine, target_table, target_year, index_columns)
    
    # clean
    del df_year_agg, df_year, availability_level_5
    collected = gc.collect()
    print(collected)


In [ ]:
log_msg("Data loaded successfully..")